In [1]:
#installing dependencies

%pip install anthropic python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# Load env variables
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
# creating an API client

from anthropic import Anthropic

client = Anthropic()
model = "claude-sonnet-5"

In [4]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system = None):
    
    params = {
        "model": model,
        "max_tokens": 1024,
        "messages": messages,
    }

    if system:
        params["system"] = system

    resp = client.messages.create(**params)

    # Normalize possible response shapes and extract text pieces
    content = None
    if hasattr(resp, 'content'):
        content = resp.content
    elif isinstance(resp, dict):
        content = resp.get('content')

    text_parts = []
    if isinstance(content, list):
        for item in content:
            # item may be a plain string
            if isinstance(item, str):
                text_parts.append(item)
                continue

            # try attribute access
            if hasattr(item, 'text'):
                try:
                    text_parts.append(getattr(item, 'text'))
                    continue
                except Exception:
                    pass

            # try dict-style access
            if isinstance(item, dict) and 'text' in item:
                text_parts.append(item['text'])
                continue

            # last resort: stringify the item
            text_parts.append(str(item))

    else:
        # Fallbacks if `content` isn't a list
        if hasattr(resp, 'text'):
            return getattr(resp, 'text')
        if isinstance(resp, dict) and 'text' in resp:
            return resp['text']
        # As a final fallback, return the string representation
        return str(resp)

    return ''.join(text_parts)

In [ ]:
messages = []

add_user_message(messages, "Write a 1 sentence description of a fake database")

stream = client.messages.create(
    model = model,
    max_tokens = 1000,
    messages = messages,
    stream = True
)

for event in stream:
    print(event)

RawMessageStartEvent(message=Message(id='msg_011CdJ7RCF3NbVow3p4ZrEjc', container=None, content=[], model='claude-sonnet-5', role='assistant', stop_details=None, stop_reason=None, stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='global', input_tokens=21, output_tokens=1, output_tokens_details=None, server_tool_use=None, service_tier='standard')), type='message_start')
RawContentBlockStartEvent(content_block=TextBlock(citations=None, text='', type='text'), index=0, type='content_block_start')
RawContentBlockDeltaEvent(delta=TextDelta(text='A', type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(delta=TextDelta(text=' cloud-based inventory management database called "StockWise" that tracks real-time product qu', type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(delta=

In [ ]:
messages = []

add_user_message(messages, "Write a 1 sentence description of a fake database")

with client.messages.stream(
    model = model,
    max_tokens = 1000,
    messages = messages
) as stream:
    for text in stream.text_stream:
        print(text, end="")